# Visualización de datos preprocesados

Lee los datos de `output/preprocessed/{train,val,test}/` generados por `preprocessing_pipeline.py` y permite verificar visualmente que el preprocesamiento es correcto.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
output_dir = "output"
split = "train"  # Cambiar a "val" o "test" para ver otros splits

split_dir = os.path.join(output_dir, "preprocessed", split)
meta = pd.read_csv(os.path.join(split_dir, "metadata.csv"))

print(f"Split: {split}")
print(f"Nódulos: {len(meta)}")
print(f"Pacientes: {meta.patient_id.nunique()}")
meta.head(10)

## 1. Estadísticas de features por split

In [ ]:
features = ["malignancy", "subtlety", "sphericity", "margin",
            "lobulation", "spiculation", "texture"]
available = [f for f in features if f in meta.columns]
meta[available].describe().round(2)

## 2. Distribución de malignancy por split

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, s in enumerate(["train", "val", "test"]):
    csv_path = os.path.join(output_dir, "preprocessed", s, "metadata.csv")
    if not os.path.exists(csv_path):
        axes[i].set_title(f"{s} — no encontrado")
        continue
    m = pd.read_csv(csv_path)
    axes[i].hist(m["malignancy"], bins=15, edgecolor="black", alpha=0.7)
    axes[i].set_xlabel("Malignancy")
    axes[i].set_ylabel("Nº nódulos")
    axes[i].set_title(f"{s} (n={len(m)})")

plt.suptitle("Distribución de malignancy por split", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Visualizar un nódulo preprocesado

Cambia `nodule_row` para explorar otros nódulos.

In [ ]:
nodule_row = 0  # <-- cambia este valor

row = meta.iloc[nodule_row]
fname = f"{row.patient_id}_nod{int(row.nodule_idx)}.npy"

ct = np.load(os.path.join(split_dir, "CT", fname))
mask = np.load(os.path.join(split_dir, "masks", fname))

print(f"Paciente: {row.patient_id}")
print(f"Nódulo: {int(row.nodule_idx)}")
print(f"Shape: {ct.shape}")
print(f"Rango CT: [{ct.min():.3f}, {ct.max():.3f}]")
print(f"Dtype CT: {ct.dtype}")
print(f"Valores máscara: {np.unique(mask)}")
print(f"Malignancy: {row.malignancy}")

In [ ]:
# Slice central: CT, máscara y overlay
z = ct.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(ct[:, :, z], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("CT preprocesado")
axes[0].axis("off")

axes[1].imshow(mask[:, :, z], cmap="gray")
axes[1].set_title("Máscara")
axes[1].axis("off")

axes[2].imshow(ct[:, :, z], cmap="gray", vmin=0, vmax=1)
axes[2].imshow(mask[:, :, z], alpha=0.4, cmap="Reds")
axes[2].set_title("CT + Máscara")
axes[2].axis("off")

plt.suptitle(f"{row.patient_id} — nod{int(row.nodule_idx)} — mal={row.malignancy}", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Todos los slices del nódulo

In [ ]:
n_slices = ct.shape[2]
cols = min(n_slices, 8)
rows_grid = (n_slices + cols - 1) // cols

fig, axes = plt.subplots(rows_grid, cols, figsize=(3 * cols, 3 * rows_grid))
axes = np.atleast_2d(axes)

for i in range(rows_grid * cols):
    r, c = divmod(i, cols)
    ax = axes[r, c]
    if i < n_slices:
        ax.imshow(ct[:, :, i], cmap="gray", vmin=0, vmax=1)
        ax.imshow(mask[:, :, i], alpha=0.4, cmap="Reds")
        ax.set_title(f"z={i}", fontsize=9)
    ax.axis("off")

plt.suptitle(f"{row.patient_id} — nod{int(row.nodule_idx)} — todos los slices", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Comparar varios nódulos del split

In [ ]:
n_show = min(8, len(meta))
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))

for i in range(n_show):
    r = meta.iloc[i]
    f = f"{r.patient_id}_nod{int(r.nodule_idx)}.npy"
    ct_i = np.load(os.path.join(split_dir, "CT", f))
    mask_i = np.load(os.path.join(split_dir, "masks", f))
    z = ct_i.shape[2] // 2

    axes[0, i].imshow(ct_i[:, :, z], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"mal={r.malignancy}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(ct_i[:, :, z], cmap="gray", vmin=0, vmax=1)
    axes[1, i].imshow(mask_i[:, :, z], alpha=0.4, cmap="Reds")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("CT", fontsize=11)
axes[1, 0].set_ylabel("CT + Máscara", fontsize=11)
plt.suptitle(f"Split: {split} — comparación de nódulos (slice central)", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Buscar por paciente

In [ ]:
patient_id = "LIDC-IDRI-0078"  # <-- cambia el paciente

# Buscar en todos los splits
for s in ["train", "val", "test"]:
    csv_path = os.path.join(output_dir, "preprocessed", s, "metadata.csv")
    if not os.path.exists(csv_path):
        continue
    m = pd.read_csv(csv_path)
    paciente = m[m.patient_id == patient_id]
    if len(paciente) > 0:
        print(f"Encontrado en split: {s}")
        print(f"Nódulos: {len(paciente)}")
        print(paciente[["patient_id", "nodule_idx", "num_annotations", "malignancy"]].to_string(index=False))
        
        s_dir = os.path.join(output_dir, "preprocessed", s)
        for _, pr in paciente.iterrows():
            fn = f"{pr.patient_id}_nod{int(pr.nodule_idx)}.npy"
            ct_p = np.load(os.path.join(s_dir, "CT", fn))
            mask_p = np.load(os.path.join(s_dir, "masks", fn))
            zz = ct_p.shape[2] // 2
            
            fig, ax = plt.subplots(1, 3, figsize=(15, 5))
            ax[0].imshow(ct_p[:, :, zz], cmap="gray", vmin=0, vmax=1)
            ax[0].set_title("CT")
            ax[0].axis("off")
            ax[1].imshow(mask_p[:, :, zz], cmap="gray")
            ax[1].set_title("Máscara")
            ax[1].axis("off")
            ax[2].imshow(ct_p[:, :, zz], cmap="gray", vmin=0, vmax=1)
            ax[2].imshow(mask_p[:, :, zz], alpha=0.4, cmap="Reds")
            ax[2].set_title("Overlay")
            ax[2].axis("off")
            plt.suptitle(f"{pr.patient_id} nod{int(pr.nodule_idx)} — mal={pr.malignancy}", fontsize=13)
            plt.tight_layout()
            plt.show()
        break
else:
    print(f"Paciente {patient_id} no encontrado en ningún split")